In [1]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,SystemMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langgraph.graph import StateGraph,START, END,MessagesState
from typing import TypedDict,Annotated,Literal
import os 
import sys
sys.path.insert(1, r'D:\Notebooks\LLM\env')
from enviorment import load_env
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.tools import tool
#from pydirectoryloader import rag_function
import os 
load_env()
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import GoogleSerperAPIWrapper,WikipediaAPIWrapper
import logging
from langchain.agents.middleware import before_model, after_model
from langchain.agents.middleware import SummarizationMiddleware
#from langchain.middleware.summarization import SummarizationMiddleware
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
#from langchain_anthropic.middleware import anthropicPromptCachingMiddleware
from langchain.agents.middleware import TodoListMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
api_key='5c55dcd133444b748d50d1255a622983'
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY", "5c55dcd133444b748d50d1255a622983")

In [3]:
from pageindex import PageIndexClient

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)


In [17]:
PDF_URL = "https://www.orimi.com/pdf-test.pdf"
DOWNLOAD_DIR = "./data"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

In [18]:
pdf_path = os.path.join(DOWNLOAD_DIR, PDF_URL.split("/")[-1])
pdf_path

'./data\\pdf-test.pdf'

In [19]:
import requests
import time
import pageindex.utils as utils

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
pdf_path = os.path.join(DOWNLOAD_DIR, PDF_URL.split("/")[-1])
if not os.path.exists(pdf_path):
        print("Downloading", PDF_URL)
        r = requests.get(PDF_URL, timeout=30)
        r.raise_for_status()
        with open(pdf_path, "wb") as f:
            f.write(r.content)
        print("Saved to", pdf_path)
else:
    print("PDF already present:", pdf_path)
    # Step 1: Submit to PageIndex for tree generation
print("Submitting document to PageIndex...")
submit_resp = pi_client.submit_document(pdf_path)
# Notebook examples return {"doc_id": "..."}
doc_id = submit_resp.get("doc_id") or submit_resp.get("id") or submit_resp
print("Submitted. doc_id:", doc_id)
print("Waiting for PageIndex tree generation (polling)...")
for i in range(60):  # up to ~10 minutes depending on doc + service
    ready = pi_client.is_retrieval_ready(doc_id)
    if ready:
        print("Tree ready")
        break
        print(f"Not ready yet... sleeping 5s ({i+1}/60)")
        time.sleep(5)
tree_resp = pi_client.get_tree(doc_id, node_summary=True)
    # Many notebook examples return {'result': tree}
tree = tree_resp.get("result") if isinstance(tree_resp, dict) and "result" in tree_resp else tree_resp
print("Fetched tree. Top-level nodes:", len(tree))
# Build a node map for easy lookup
node_map = utils.create_node_mapping(tree)

PDF already present: ./data\pdf-test.pdf
Submitting document to PageIndex...
Submitted. doc_id: pi-cmmzyfdf900fb0ipk7g32yzjm
Waiting for PageIndex tree generation (polling)...
Tree ready
Fetched tree. Top-level nodes: 1


In [21]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [23]:
import json
query = "What are the conclusions in this document?"
    # Remove large text fields to keep the prompt small
tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])
search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.
Question: {query}
Document tree structure:
{json.dumps(tree_without_text, indent=2)}
Please reply in the following JSON format:
{{
"thinking": "<Your thinking process on which nodes are relevant to the question>",
"node_list": ["node_id_1", "node_id_2"]
}}
Directly return the final JSON structure. Do not output anything else.
"""
print("Asking LLM to search the tree for relevant nodes...")
tree_search_result_text = llm.invoke(search_prompt)
#tree_search_result = json.loads(tree_search_result_text)#
#print("\n=== LLM reasoning (condensed) ===")
#print(tree_search_result.get("thinking", "")[:1000], "\n")
#print("Node IDs returned:", tree_search_result.get("node_list"))

Asking LLM to search the tree for relevant nodes...


In [ ]:
tree

[{'title': 'PDF Test File',
  'node_id': '0000',
  'page_index': 1,
  'summary': '# PDF Test File\n\nCongratulations, your computer is equipped with a PDF (Portable Document Format) reader! You should be able to view any of the PDF documents and forms available on our site. PDF forms are indicated by these icons: ▲ or ▼.\n\nYukon Department of Education\nBox 2703\nWhitehorse,Yukon\nCanada\nY1A 2C6\n\nPlease visit our website at: http://www.education.gov.yk.ca/\n',
  'text': '# PDF Test File\n\nCongratulations, your computer is equipped with a PDF (Portable Document Format) reader! You should be able to view any of the PDF documents and forms available on our site. PDF forms are indicated by these icons: ▲ or ▼.\n\nYukon Department of Education\nBox 2703\nWhitehorse,Yukon\nCanada\nY1A 2C6\n\nPlease visit our website at: http://www.education.gov.yk.ca/\n'}]

: 

In [27]:
eval(tree_search_result_text.content)['node_list']

['0000']